# Aggregasi & Business Insight per Versi Aplikasi
Perhitungan frekuensi aspek dan persentase komplain (sentimen negatif) di tingkat versi aplikasi dan waktu (bulanan) untuk laporan visualisasi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
dataset_path = '../data/reviews_with_aspect_sentiment.csv'
df = pd.read_csv(dataset_path)
print(f"Total data: {len(df)} baris")

## 2. Hitung Agregasi Aspek per Versi Aplikasi & Bulanan

In [ ]:
aspect_cols = ['aspek_rasa', 'aspek_harga', 'aspek_aplikasi', 'aspek_pelayanan', 'aspek_kecepatan', 'aspek_stok', 'aspek_pengiriman', 'aspek_pesanan']

# 1. Handle missing app versions
df['app_version'] = df['app_version'].fillna('Unknown')

# 2. Extract monthly period
df['date_dt'] = pd.to_datetime(df['date'])
df['year_month'] = df['date_dt'].dt.to_period('M').astype(str)

def aggregate_by_dim(dim_col):
    base = df.groupby(dim_col).agg(
        total_review=('review_text', 'count'),
        avg_rating=('rating', 'mean')
    ).reset_index()
    
    aspects = df.groupby(dim_col)[aspect_cols].sum().reset_index()
    
    def calc_neg_ratios(group):
        ratios = {}
        for aspect in aspect_cols:
            sentiment_col = aspect.replace('aspek_', 'sentimen_')
            aspect_mentions = group[aspect].sum()
            neg_mentions = (group[sentiment_col] == 'negative').sum()
            ratios[aspect.replace('aspek_', 'neg_ratio_')] = neg_mentions / aspect_mentions if aspect_mentions > 0 else 0.0
        return pd.Series(ratios)
        
    neg_ratios = df.groupby(dim_col).apply(calc_neg_ratios).reset_index()
    
    merged = pd.merge(base, aspects, on=dim_col)
    merged = pd.merge(merged, neg_ratios, on=dim_col)
    return merged

agg_version = aggregate_by_dim('app_version')
agg_month = aggregate_by_dim('year_month')

## 3. Identifikasi Masalah Terbesar per Versi Aplikasi

In [ ]:
# Tampilkan versi aplikasi dengan komplain performa aplikasi (aspek_aplikasi) tertinggi
high_volume_versions = agg_version[agg_version['total_review'] >= 50]
top_app_issues = high_volume_versions.sort_values(by='neg_ratio_aplikasi', ascending=False).head(5)

print("=== Top 5 Versi Aplikasi dengan Rasio Komplain Aplikasi Tertinggi (Min. 50 Review) ===")
for _, row in top_app_issues.iterrows():
    print(f"Versi: {row['app_version']} | Total Review: {row['total_review']} | Rasio Negatif: {row['neg_ratio_aplikasi']*100:.2f}%")

## 4. Visualisasi Tren Komplain Bulanan

In [ ]:
plt.figure(figsize=(12, 6))
sorted_months = agg_month.sort_values(by='year_month')
plt.plot(sorted_months['year_month'], sorted_months['neg_ratio_aplikasi'] * 100, marker='o', label='Rasio Negatif Aplikasi', color='red')
plt.plot(sorted_months['year_month'], sorted_months['avg_rating'] * 20, marker='x', label='Skor Rating (Skala 0-100)', color='blue')
plt.title('Tren Bulanan: Rating Aplikasi vs Rasio Komplain Teknis (%)')
plt.xlabel('Bulan')
plt.ylabel('Skor (%)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Simpan Hasil Agregasi

In [ ]:
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/aspect_analysis_app.csv', index=False)
agg_version.to_csv('../data/aggregated_insight_per_version.csv', index=False)
agg_month.to_csv('../data/aggregated_insight_per_month.csv', index=False)
print("Data agregasi berhasil disimpan di modeling/data/")